# 02 - Feature Engineering

Builds rolling 10-game (min 3) averages of forecheck and baseline stats, then 
reshapes from one row per team-game to one row per game (home/away side by side).

Outputs two training sets from 2008-2024 data: full history and recent era 
(2017-2024). The 2025-26 test set isn't included here - it needs separate 
cleaning once that data is available.

In [1]:
import pandas as pd

team_game_collapsed = pd.read_csv('../data/processed/team_game_collapsed.csv')
print(team_game_collapsed.shape)
print(team_game_collapsed.columns.tolist())

(43632, 117)
['team', 'season', 'name', 'gameId', 'playerTeam', 'opposingTeam', 'home_or_away', 'gameDate', 'position', 'situation', 'xGoalsPercentage', 'corsiPercentage', 'fenwickPercentage', 'iceTime', 'xOnGoalFor', 'xGoalsFor', 'xReboundsFor', 'xFreezeFor', 'xPlayStoppedFor', 'xPlayContinuedInZoneFor', 'xPlayContinuedOutsideZoneFor', 'flurryAdjustedxGoalsFor', 'scoreVenueAdjustedxGoalsFor', 'flurryScoreVenueAdjustedxGoalsFor', 'shotsOnGoalFor', 'missedShotsFor', 'blockedShotAttemptsFor', 'shotAttemptsFor', 'goalsFor', 'reboundsFor', 'reboundGoalsFor', 'freezeFor', 'playStoppedFor', 'playContinuedInZoneFor', 'playContinuedOutsideZoneFor', 'savedShotsOnGoalFor', 'savedUnblockedShotAttemptsFor', 'penaltiesFor', 'penalityMinutesFor', 'faceOffsWonFor', 'hitsFor', 'takeawaysFor', 'giveawaysFor', 'lowDangerShotsFor', 'mediumDangerShotsFor', 'highDangerShotsFor', 'lowDangerxGoalsFor', 'mediumDangerxGoalsFor', 'highDangerxGoalsFor', 'lowDangerGoalsFor', 'mediumDangerGoalsFor', 'highDangerGoa

In [2]:
team_game_collapsed['save_percentage'] = (
    team_game_collapsed['savedShotsOnGoalAgainst'] / team_game_collapsed['shotsOnGoalAgainst']
)

print(team_game_collapsed[['team', 'opposingTeam', 'gameId', 'savedShotsOnGoalAgainst', 'shotsOnGoalAgainst', 'save_percentage']].head())
print(team_game_collapsed['save_percentage'].isna().sum(), 'NaN values')

  team opposingTeam      gameId  savedShotsOnGoalAgainst  shotsOnGoalAgainst  \
0  NYR          TBL  2008020001                     20.0                21.0   
1  NYR          TBL  2008020003                     18.0                19.0   
2  NYR          CHI  2008020010                     30.0                32.0   
3  NYR          PHI  2008020019                     25.0                28.0   
4  NYR          NJD  2008020034                     26.0                27.0   

   save_percentage  
0         0.952381  
1         0.947368  
2         0.937500  
3         0.892857  
4         0.962963  
0 NaN values


/var/folders/dn/3nxcdfzn4z94v26m70j39pdc0000gn/T/ipykernel_23791/3817766877.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_game_collapsed['save_percentage'] = (


In [3]:
team_game_collapsed = team_game_collapsed.sort_values(['team', 'gameDate']).reset_index(drop=True)

print(team_game_collapsed[['team', 'gameDate', 'gameId']].head(15))

   team  gameDate      gameId
0   ANA  20081009  2008020008
1   ANA  20081012  2008020030
2   ANA  20081014  2008020042
3   ANA  20081015  2008020048
4   ANA  20081017  2008020061
5   ANA  20081019  2008020077
6   ANA  20081021  2008020083
7   ANA  20081024  2008020099
8   ANA  20081025  2008020107
9   ANA  20081027  2008020122
10  ANA  20081029  2008020135
11  ANA  20081031  2008020149
12  ANA  20081102  2008020166
13  ANA  20081104  2008020176
14  ANA  20081105  2008020180


In [ ]:
# REPLACE WITH ACTUAL FEATURES

rolling_cols = [
    'forecheck_turnover_index', 
    'forecheck_turnover_index_z', 
    'total_pressure_index_z', 
    'win', 
    'corsiPercentage', 
    'save_percentage'
]

for col in rolling_cols:
    team_game_collapsed[f'{col}_rolling10'] = (
        team_game_collapsed.groupby('team')[col]
        .transform(lambda x: x.shift(1).rolling(window=10, min_periods=3).mean())
    )

print(team_game_collapsed[['team', 'gameDate', 'win', 'win_rolling10']].head(15))